# Language Encoding for Robots: Three Levels

**Lecture 2, Part 1 — Vizuara Modern Robot Learning Bootcamp**

In this notebook, we encode the **same robot instruction** using three different techniques:

| Level | Method | Key Idea |
|-------|--------|----------|
| 0 | Bag of Words | Count word occurrences — sparse, no semantics |
| 1 | Word Embeddings | Dense learned vectors — semantic similarity |
| 2 | GRU | Sequential reading — sentence-level embedding |

By the end, you'll see **why GRU is the minimum viable approach** for encoding robot instructions — and why even it has serious limitations.

---

In [ ]:
# Install dependencies (Colab already has these, but just in case)
!pip install -q torch numpy matplotlib 2>&1 | tail -3

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

# Vizuara color palette
ACCENT = '#D97757'
BLUE   = '#5B8CB8'
TEAL   = '#7DA488'
PURPLE = '#9B7EC8'
WARM   = '#C4956A'
BG     = '#FAF8F5'

plt.rcParams['figure.facecolor'] = BG
plt.rcParams['axes.facecolor'] = BG
plt.rcParams['font.size'] = 12

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## Our Robot Instructions

We'll encode these 6 sentences — some mean the same thing, some don't:

In [ ]:
sentences = [
    "pick up the red cup",
    "grab the crimson mug",        # same meaning, different words
    "put down the blue plate",     # different action + object
    "lift the red cup gently",     # similar to sentence 0
    "push the cube to the goal",   # completely different task
    "pick up the red cup",         # exact duplicate of sentence 0
]

labels = [
    '"pick up the red cup"',
    '"grab the crimson mug"',
    '"put down the blue plate"',
    '"lift the red cup gently"',
    '"push the cube to the goal"',
    '"pick up the red cup" (dup)',
]

for i, s in enumerate(sentences):
    print(f"  [{i}] {s}")

---
## Level 0: Bag of Words

Build a vocabulary from all sentences, then represent each sentence as a **count vector**.

In [ ]:
# Build vocabulary
all_words = set()
for s in sentences:
    all_words.update(s.lower().split())
vocab_list = sorted(all_words)
word_to_idx = {w: i for i, w in enumerate(vocab_list)}

print(f"Vocabulary ({len(vocab_list)} words): {vocab_list}")
print(f"\nEach sentence → a vector of length {len(vocab_list)}")

In [ ]:
def bag_of_words(sentence, word_to_idx):
    """Convert a sentence to a BoW count vector."""
    vec = np.zeros(len(word_to_idx))
    for word in sentence.lower().split():
        if word in word_to_idx:
            vec[word_to_idx[word]] += 1
    return vec

bow_vectors = np.array([bag_of_words(s, word_to_idx) for s in sentences])

# Show the vectors
print("Bag of Words vectors:")
print(f"{'':>36s} | {' '.join(f'{w:>6s}' for w in vocab_list)}")
print("-" * (37 + 7 * len(vocab_list)))
for i, s in enumerate(sentences):
    vals = ' '.join(f'{int(v):>6d}' for v in bow_vectors[i])
    print(f"{s:>35s}  | {vals}")

In [ ]:
def cosine_similarity_matrix(vectors):
    """Compute pairwise cosine similarity."""
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1, norms)  # avoid division by zero
    normed = vectors / norms
    return normed @ normed.T

bow_sim = cosine_similarity_matrix(bow_vectors)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(bow_sim, cmap='RdYlGn', vmin=0, vmax=1)
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels([f'[{i}]' for i in range(len(labels))], fontsize=10)
ax.set_yticklabels(labels, fontsize=9)
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f'{bow_sim[i,j]:.2f}', ha='center', va='center', fontsize=9)
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title('Bag of Words — Cosine Similarity', fontsize=14, color=ACCENT, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\n⚠️  "pick up the red cup" vs "grab the crimson mug" similarity: {bow_sim[0,1]:.2f}')
print(f'    These mean the SAME thing but BoW sees almost no overlap!')

---
## Level 1: Word Embeddings

Instead of sparse one-hot vectors, we learn **dense vectors** where similar words are nearby.

We'll use `nn.Embedding` — a learnable lookup table. In practice, these would be pre-trained (like Word2Vec/GloVe), but here we'll train them from scratch to see the concept.

In [ ]:
# Tokenize sentences
def tokenize(sentence, word_to_idx):
    return [word_to_idx[w] for w in sentence.lower().split() if w in word_to_idx]

token_ids = [tokenize(s, word_to_idx) for s in sentences]
print("Tokenized:")
for i, (s, t) in enumerate(zip(sentences, token_ids)):
    print(f"  [{i}] {s} → {t}")

In [ ]:
# Create embedding layer
EMBED_DIM = 32
embedding = nn.Embedding(len(vocab_list), EMBED_DIM)

# Get per-word embeddings and average them for a sentence vector
# (This is the simplest sentence embedding: mean of word embeddings)
def embed_sentence_avg(token_ids_list, embedding_layer):
    """Average word embeddings to get sentence embedding."""
    vecs = []
    for tids in token_ids_list:
        t = torch.tensor(tids, dtype=torch.long)
        word_vecs = embedding_layer(t)  # [seq_len, embed_dim]
        vecs.append(word_vecs.mean(dim=0))  # [embed_dim]
    return torch.stack(vecs).detach().numpy()

# With RANDOM (untrained) embeddings
embed_vecs_random = embed_sentence_avg(token_ids, embedding)
embed_sim_random = cosine_similarity_matrix(embed_vecs_random)

print(f"Embedding dim: {EMBED_DIM}")
print(f"Sentence embedding shape: {embed_vecs_random.shape}")
print(f"\nWith RANDOM embeddings, similarity between [0] and [1]: {embed_sim_random[0,1]:.2f}")
print("(Still random — embeddings need training to be meaningful!)")

In [ ]:
# Let's simulate trained embeddings by manually making synonyms close
# This demonstrates the CONCEPT of what Word2Vec would learn

with torch.no_grad():
    # Make action synonyms similar
    base_action = torch.randn(EMBED_DIM)
    for w in ['pick', 'grab', 'lift']:
        embedding.weight[word_to_idx[w]] = base_action + torch.randn(EMBED_DIM) * 0.1

    # Make object synonyms similar
    base_obj = torch.randn(EMBED_DIM)
    for w in ['cup', 'mug']:
        embedding.weight[word_to_idx[w]] = base_obj + torch.randn(EMBED_DIM) * 0.1

    # Make color synonyms similar
    base_color = torch.randn(EMBED_DIM)
    for w in ['red', 'crimson']:
        embedding.weight[word_to_idx[w]] = base_color + torch.randn(EMBED_DIM) * 0.1

embed_vecs_trained = embed_sentence_avg(token_ids, embedding)
embed_sim_trained = cosine_similarity_matrix(embed_vecs_trained)

# Plot both side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, sim, title in zip(axes,
    [embed_sim_random, embed_sim_trained],
    ['Word Embeddings (Random/Untrained)', 'Word Embeddings (Simulated Pre-trained)']):
    im = ax.imshow(sim, cmap='RdYlGn', vmin=0, vmax=1)
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels([f'[{i}]' for i in range(len(labels))], fontsize=9)
    ax.set_yticklabels(labels, fontsize=8)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, f'{sim[i,j]:.2f}', ha='center', va='center', fontsize=8)
    ax.set_title(title, fontsize=12, color=ACCENT, fontweight='bold')
plt.colorbar(im, ax=axes, shrink=0.8)
plt.tight_layout()
plt.show()

print(f'With simulated pre-training:')
print(f'  "pick up the red cup" vs "grab the crimson mug": {embed_sim_trained[0,1]:.2f} ← MUCH higher!')
print(f'  "pick up the red cup" vs "push the cube to the goal": {embed_sim_trained[0,4]:.2f} ← low (different task)')

### Problem: Averaging Destroys Word Order

With average pooling, "the dog bit the man" and "the man bit the dog" get **identical** embeddings!

In [ ]:
# Demonstrate the word-order problem
order_sentences = [
    "put the cup on the plate",
    "put the plate on the cup",
]

# Add missing words to vocab for this demo
for s in order_sentences:
    for w in s.split():
        if w not in word_to_idx:
            word_to_idx[w] = len(word_to_idx)
            vocab_list.append(w)

# Resize embedding
new_embedding = nn.Embedding(len(vocab_list), EMBED_DIM)
with torch.no_grad():
    new_embedding.weight[:embedding.weight.shape[0]] = embedding.weight

order_tokens = [tokenize(s, word_to_idx) for s in order_sentences]
order_vecs = embed_sentence_avg(order_tokens, new_embedding)
sim = cosine_similarity_matrix(order_vecs)

print(f'Sentence A: "{order_sentences[0]}"')
print(f'Sentence B: "{order_sentences[1]}"')
print(f'\nCosine similarity (averaged embeddings): {sim[0,1]:.4f}')
print(f'\n⚠️  Similarity ≈ 1.0 — these mean OPPOSITE things but look identical!')
print(f'    Averaging destroys word order. We need something that reads sequentially.')

---
## Level 2: GRU — Sequential Sentence Encoding

A GRU reads words **one by one**, maintaining a hidden state. The final hidden state encodes the whole sentence.

In [ ]:
class TextEncoderGRU(nn.Module):
    """GRU-based text encoder: sentence → fixed-size vector."""
    def __init__(self, vocab_size, d_word=32, d_model=64):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_word)
        self.gru = nn.GRU(d_word, d_model, batch_first=True)
        self.ln = nn.LayerNorm(d_model)

    def forward(self, token_ids):
        # token_ids: [batch, seq_len]
        x = self.embed(token_ids)       # [batch, seq_len, d_word]
        _, h_last = self.gru(x)         # h_last: [1, batch, d_model]
        h = h_last[0]                   # [batch, d_model]
        return self.ln(h)

# Build the encoder
gru_encoder = TextEncoderGRU(vocab_size=len(vocab_list), d_word=32, d_model=64)
gru_encoder.eval()

print(f"GRU Text Encoder:")
print(f"  Vocab size: {len(vocab_list)}")
print(f"  Word embedding dim: 32")
print(f"  Sentence embedding dim: 64")
total_params = sum(p.numel() for p in gru_encoder.parameters())
print(f"  Total parameters: {total_params:,}")

In [ ]:
# Pad sequences to same length and encode
def pad_sequences(token_lists, pad_value=0):
    max_len = max(len(t) for t in token_lists)
    padded = [t + [pad_value] * (max_len - len(t)) for t in token_lists]
    return torch.tensor(padded, dtype=torch.long)

# Re-tokenize with our expanded vocab
token_ids_all = [tokenize(s, word_to_idx) for s in sentences]
padded = pad_sequences(token_ids_all)

with torch.no_grad():
    gru_vecs = gru_encoder(padded).numpy()

print(f"GRU output shape: {gru_vecs.shape}")
print(f"Each sentence → one 64-d vector, regardless of length")

In [ ]:
# GRU similarity (untrained — random initialization)
gru_sim = cosine_similarity_matrix(gru_vecs)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(gru_sim, cmap='RdYlGn', vmin=-1, vmax=1)
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels([f'[{i}]' for i in range(len(labels))], fontsize=10)
ax.set_yticklabels(labels, fontsize=9)
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f'{gru_sim[i,j]:.2f}', ha='center', va='center', fontsize=9)
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title('GRU Encoder — Cosine Similarity (Untrained)', fontsize=14, color=ACCENT, fontweight='bold')
plt.tight_layout()
plt.show()

print("⚠️  With RANDOM weights, the GRU doesn't produce meaningful similarities yet.")
print("   But notice: identical sentences [0] and [5] get similarity 1.00 — the GRU IS")
print("   sensitive to word order (unlike averaged embeddings).")

In [ ]:
# Demonstrate that GRU IS order-sensitive (unlike averaged embeddings)
order_tokens_gru = [tokenize(s, word_to_idx) for s in order_sentences]
order_padded = pad_sequences(order_tokens_gru)

with torch.no_grad():
    order_gru_vecs = gru_encoder(order_padded).numpy()

order_gru_sim = cosine_similarity_matrix(order_gru_vecs)

print(f'Sentence A: "{order_sentences[0]}"')
print(f'Sentence B: "{order_sentences[1]}"')
print(f'\nGRU cosine similarity: {order_gru_sim[0,1]:.4f}')
print(f'Avg embeddings similarity: {sim[0,1]:.4f}')
print(f'\n✅ GRU gives DIFFERENT embeddings for different word orders!')
print(f'   (Avg embeddings gave ~1.0 — they were blind to order.)')

## Visualize: Hidden State Evolution

Let's watch how the GRU's hidden state changes as it reads each word.

In [ ]:
# Trace hidden states word by word
sentence = "pick up the red cup"
words = sentence.split()
tids = tokenize(sentence, word_to_idx)

hidden_states = []
h = None
with torch.no_grad():
    for tid in tids:
        x = gru_encoder.embed(torch.tensor([[tid]]))  # [1, 1, d_word]
        _, h = gru_encoder.gru(x, h)                   # h: [1, 1, d_model]
        hidden_states.append(h[0, 0].numpy().copy())

hidden_states = np.array(hidden_states)  # [5, 64]

# Plot hidden state evolution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: heatmap of hidden states
ax = axes[0]
im = ax.imshow(hidden_states.T, aspect='auto', cmap='coolwarm', interpolation='nearest')
ax.set_xticks(range(len(words)))
ax.set_xticklabels(words, fontsize=12, fontweight='bold')
ax.set_ylabel('Hidden State Dimension', fontsize=11)
ax.set_xlabel('Word (left to right)', fontsize=11)
ax.set_title('GRU Hidden State After Each Word', fontsize=13, color=ACCENT, fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.8)

# Right: L2 norm of hidden state (growing "knowledge")
ax = axes[1]
norms = np.linalg.norm(hidden_states, axis=1)
colors = [ACCENT, ACCENT, WARM, TEAL, BLUE]
ax.bar(range(len(words)), norms, color=colors, edgecolor='white', linewidth=1.5)
ax.set_xticks(range(len(words)))
ax.set_xticklabels(words, fontsize=12, fontweight='bold')
ax.set_ylabel('Hidden State Magnitude (L2 norm)', fontsize=11)
ax.set_title('Information Accumulation', fontsize=13, color=ACCENT, fontweight='bold')
for i, (n, w) in enumerate(zip(norms, words)):
    ax.text(i, n + 0.05, f'{n:.2f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

print("The hidden state changes with every word — the GRU accumulates context.")
print("After 'cup', it holds the full sentence meaning in a 64-d vector.")

---
## Side-by-Side Comparison: All Three Levels

Let's put it all together and compare the three encoding approaches.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
titles = ['Level 0: Bag of Words', 'Level 1: Word Embeddings\n(simulated pre-trained)', 'Level 2: GRU\n(untrained)']
sims = [bow_sim, embed_sim_trained, gru_sim]
vmins = [0, 0, -1]

for ax, sim_mat, title, vmin in zip(axes, sims, titles, vmins):
    im = ax.imshow(sim_mat, cmap='RdYlGn', vmin=vmin, vmax=1)
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels([f'[{i}]' for i in range(len(labels))], fontsize=9)
    ax.set_yticklabels(labels, fontsize=7)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, f'{sim_mat[i,j]:.2f}', ha='center', va='center', fontsize=7)
    ax.set_title(title, fontsize=12, color=ACCENT, fontweight='bold')

plt.tight_layout()
plt.show()

print("Key observations:")
print("  Level 0 (BoW):  Synonymous sentences [0] & [1] → low similarity (0.2)")
print("  Level 1 (Embed): Synonymous sentences [0] & [1] → HIGH similarity (with pre-training)")
print("  Level 2 (GRU):   Random init — needs training. But preserves word order!")
print("\nFor our mini-VLA, we'll use the GRU trained end-to-end with the robot policy.")

---
## Summary

| Level | Method | Synonyms? | Word Order? | Sentence Vector? | Vocab Size Needed? |
|-------|--------|-----------|-------------|-------------------|--------------------|
| 0 | Bag of Words | ❌ No | ❌ No | ✅ Yes (sparse) | All words |
| 1 | Word Embeddings | ✅ Yes | ❌ No (if averaged) | ⚠️ Only if averaged | Pre-training data |
| 2 | GRU | ✅ With training | ✅ Yes | ✅ Yes (dense) | Training data |

**The GRU is our best option** — but it still has a bottleneck: the entire sentence is compressed into ONE fixed-size vector. We'll see in Part 3 how **Transformers** solve this.

---

## 🏋️ TODO Exercise

1. **Increase GRU hidden size** from 64 to 256. Does the word-order sensitivity change?
2. **Try a bidirectional GRU** (`nn.GRU(..., bidirectional=True)`). Does it capture more context?
3. **Add more synonym groups** (e.g., push/shove, table/desk) and re-run the embedding experiment.
4. **Challenge:** Can you train the GRU embeddings with a contrastive loss so that sentences with the same task get similar embeddings?